# Phân tích Spot Instance Bidding Strategy với RL

Notebook này hướng dẫn:
1. Khám phá môi trường (Environment)
2. Phân tích dữ liệu giá spot và workload
3. Training RL agent
4. Đánh giá và so sánh với baselines
5. Visualization kết quả

In [ ]:
# Import libraries
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.environment import SpotInstanceEnv, InstanceConfig, RewardConfig
from src.data_generator import SpotPriceGenerator, WorkloadGenerator, generate_sample_data
from src.baselines import AlwaysOnDemand, AlwaysSpot, ThresholdPolicy, RandomPolicy
from src.visualize import plot_spot_price_patterns, plot_workload_patterns, create_dashboard

# Cấu hình display
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Khám phá Environment

In [ ]:
# Tạo environment
env = SpotInstanceEnv(workload_pattern='spike')

print("=" * 50)
print("Environment Info:")
print("=" * 50)
print(f"Observation Space: {env.observation_space}")
print(f"Action Space: {env.action_space}")
print(f"\nAction Names:")
for i, name in env.ACTION_NAMES.items():
    print(f"  {i}: {name}")

In [ ]:
# Reset và xem observation đầu tiên
obs, info = env.reset(seed=42)

print("Initial Observation:")
features = ['current_spot_price', 'price_moving_avg', 'hour_of_day', 
            'day_of_week', 'pending_workload', 'active_instances', 'interruption_prob']
for name, value in zip(features, obs):
    print(f"  {name}: {value:.4f}")

print(f"\nInfo: {info}")

## 2. Phân tích dữ liệu giá Spot

In [ ]:
# Sinh dữ liệu mẫu
df = generate_sample_data(n_days=7, seed=42)
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
# Thống kê mô tả
df.describe()

In [ ]:
# Vẽ giá spot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Giá theo thời gian
axes[0, 0].plot(df['step'], df['spot_price'], color='#d62728', linewidth=1)
axes[0, 0].fill_between(df['step'], df['spot_price'], alpha=0.3, color='#d62728')
axes[0, 0].set_xlabel('Hour')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].set_title('Giá Spot trong 7 ngày')

# 2. Giá theo giờ
hourly_avg = df.groupby('hour')['spot_price'].agg(['mean', 'std'])
axes[0, 1].bar(hourly_avg.index, hourly_avg['mean'], yerr=hourly_avg['std'], 
               color='#d62728', alpha=0.7, capsize=3)
axes[0, 1].set_xlabel('Hour of Day')
axes[0, 1].set_ylabel('Average Price ($)')
axes[0, 1].set_title('Giá trung bình theo Giờ')

# 3. Giá theo ngày
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily_avg = df.groupby('day_of_week')['spot_price'].mean()
axes[1, 0].bar(range(7), daily_avg.values, color='#d62728', alpha=0.7)
axes[1, 0].set_xticks(range(7))
axes[1, 0].set_xticklabels(day_names)
axes[1, 0].set_xlabel('Day of Week')
axes[1, 0].set_ylabel('Average Price ($)')
axes[1, 0].set_title('Giá trung bình theo Ngày')

# 4. Phân phối giá
axes[1, 1].hist(df['spot_price'], bins=50, color='#d62728', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(df['spot_price'].mean(), color='blue', linestyle='--', label=f"Mean: ${df['spot_price'].mean():.4f}")
axes[1, 1].set_xlabel('Price ($)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Phân phối Giá')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 3. Phân tích Workload Patterns

In [ ]:
# So sánh các pattern workload
patterns = ['stable', 'spike', 'random', 'periodic']
colors = ['#2ca02c', '#d62728', '#9467bd', '#ff7f0e']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, pattern, color in zip(axes.flat, patterns, colors):
    gen = WorkloadGenerator(pattern=pattern, seed=42)
    workloads = gen.generate_workload_series(168)  # 1 tuần
    
    ax.plot(range(168), workloads, color=color, linewidth=1)
    ax.fill_between(range(168), workloads, alpha=0.3, color=color)
    ax.set_xlabel('Hour')
    ax.set_ylabel('Jobs')
    ax.set_title(f'Pattern: {pattern.capitalize()}')
    
    # Statistics
    stats = f'Mean: {np.mean(workloads):.1f}\nMax: {max(workloads)}\nStd: {np.std(workloads):.1f}'
    ax.text(0.95, 0.95, stats, transform=ax.transAxes, verticalalignment='top', 
            horizontalalignment='right', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

## 4. Chạy Episode với Random Policy

In [ ]:
# Chạy một episode với random policy
env = SpotInstanceEnv(workload_pattern='spike')
obs, info = env.reset(seed=42)

episode_reward = 0
done = False

while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    episode_reward += reward
    done = terminated or truncated

# Lấy metrics
metrics = env.get_metrics()
print("Episode Metrics (Random Policy):")
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

print(f"\nTotal Reward: {episode_reward:.2f}")

In [ ]:
# Visualize episode
history = env.get_history()
create_dashboard(history, policy_name="Random Policy", show=True)

## 5. So sánh Baselines

In [ ]:
from src.baselines import evaluate_policy, compare_policies

# Tạo environment và baselines
env = SpotInstanceEnv(workload_pattern='spike')

baselines = [
    AlwaysOnDemand(max_instances=5),
    AlwaysSpot(max_instances=5),
    ThresholdPolicy(spot_price_threshold=0.5, max_instances=5),
    RandomPolicy(seed=42)
]

# So sánh (chạy ít episodes để demo nhanh)
results = compare_policies(baselines, env, n_episodes=10, seed=42)

In [ ]:
# Tạo bảng so sánh
comparison_data = []
for name, metrics in results.items():
    comparison_data.append({
        'Policy': name,
        'Mean Cost': metrics['mean_total_cost'],
        'Mean Jobs': metrics['mean_total_jobs_completed'],
        'Cost/Job': metrics['mean_cost_per_job'],
        'Interruptions': metrics['mean_total_interruptions'],
        'Mean Reward': metrics['mean_episode_reward']
    })

df_comparison = pd.DataFrame(comparison_data)
df_comparison

In [ ]:
# Vẽ biểu đồ so sánh
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

colors = plt.cm.Set2(np.linspace(0, 1, len(df_comparison)))

# 1. Total Cost
axes[0, 0].bar(df_comparison['Policy'], df_comparison['Mean Cost'], color=colors)
axes[0, 0].set_ylabel('Total Cost ($)')
axes[0, 0].set_title('So sánh Chi phí')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Jobs Completed
axes[0, 1].bar(df_comparison['Policy'], df_comparison['Mean Jobs'], color=colors)
axes[0, 1].set_ylabel('Jobs Completed')
axes[0, 1].set_title('So sánh Jobs hoàn thành')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Cost per Job
axes[1, 0].bar(df_comparison['Policy'], df_comparison['Cost/Job'], color=colors)
axes[1, 0].set_ylabel('Cost per Job ($)')
axes[1, 0].set_title('So sánh Chi phí/Job')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Interruptions
axes[1, 1].bar(df_comparison['Policy'], df_comparison['Interruptions'], color=colors)
axes[1, 1].set_ylabel('Interruptions')
axes[1, 1].set_title('So sánh Interruptions')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Training RL Agent (Quick Demo)

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

# Tạo environment
env = SpotInstanceEnv(workload_pattern='spike', seed=42)

# Tạo model DQN
model = DQN(
    "MlpPolicy",
    env,
    learning_rate=0.0001,
    buffer_size=10000,
    learning_starts=500,
    batch_size=32,
    gamma=0.99,
    exploration_fraction=0.3,
    exploration_final_eps=0.05,
    verbose=1
)

print("Model created successfully!")
print(f"Policy network: {model.policy}")

In [ ]:
# Training (quick demo - 10000 steps)
# Tăng lên 100000+ steps để có kết quả tốt hơn
model.learn(total_timesteps=10000, progress_bar=True)
print("Training completed!")

In [ ]:
# Đánh giá model sau training
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"Trained DQN Agent:")
print(f"  Mean Reward: {mean_reward:.2f} +/- {std_reward:.2f}")

In [ ]:
# Chạy một episode với trained agent
obs, info = env.reset(seed=123)
episode_reward = 0
done = False

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    episode_reward += reward
    done = terminated or truncated

# Lấy metrics
metrics = env.get_metrics()
print("Episode Metrics (Trained DQN):")
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

print(f"\nTotal Reward: {episode_reward:.2f}")

In [ ]:
# Visualize trained agent behavior
history = env.get_history()
create_dashboard(history, policy_name="Trained DQN Agent", show=True)

## 7. Kết luận

Trong notebook này, chúng ta đã:

1. **Khám phá Environment**: Hiểu state space (7 features) và action space (5 actions)

2. **Phân tích dữ liệu**: 
   - Giá spot thay đổi theo giờ (cao vào ban ngày) và ngày (thấp cuối tuần)
   - Workload có nhiều patterns khác nhau

3. **So sánh Baselines**:
   - AlwaysOnDemand: Chi phí cao, ổn định
   - AlwaysSpot: Chi phí thấp, nhiều interruptions
   - ThresholdPolicy: Cân bằng

4. **Training DQN Agent**: 
   - Agent học cách chọn action tối ưu
   - Sau training đầy đủ, agent sẽ outperform baselines

### Bước tiếp theo

- Train model với nhiều timesteps hơn (100k+)
- Thử các hyperparameters khác nhau
- Evaluate trên nhiều workload patterns
- So sánh với các RL algorithms khác (PPO, A2C...)